In [36]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.ensemble import GradientBoostingClassifier

RSEED = 50

In [37]:
df_test = pd.read_csv("data/test.csv")
df_train = pd.read_csv('data/train.csv')

print(df_test.columns)
print(df_train.columns)

Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region'],
      dtype='object')
Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need'],
      dtype='object')


In [38]:
df_train.isna().sum()

id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64

In [39]:
test_ids = df_test["id"]  
X_test = df_test.drop(columns=["id"])
df = df_train.drop(columns=["id"])

In [40]:
# map target 
target = "Irrigation_Need"

target_map = {"Low": 0, "Medium": 1, "High": 2}
df[target] = df[target].map(target_map)

In [41]:
X_train = df.drop(columns=["Irrigation_Need"])
y_train = df["Irrigation_Need"]

In [42]:
binary_features = ["Mulching_Used"]

categorical_features = [
    "Soil_Type",
    "Crop_Type",
    "Irrigation_Type",
    "Water_Source",
    "Region",
    "Crop_Growth_Stage",
    "Season"
]

numeric_features = [col for col in X_train.columns if col not in categorical_features + binary_features]

print(1+len(binary_features) + len(numeric_features) + len(categorical_features))

20


In [43]:
# encoding categorical features

mulching_map = {"Yes": 1, "No": 0}
X_train["Mulching_Used"] = X_train["Mulching_Used"].map(mulching_map)
X_test["Mulching_Used"] = X_test["Mulching_Used"].map(mulching_map)

ohe = OneHotEncoder(drop="first", sparse=False, handle_unknown="ignore")
X_train_cat = ohe.fit_transform(X_train[categorical_features])
X_test_cat = ohe.transform(X_test[categorical_features])

cat_cols = ohe.get_feature_names_out(categorical_features)

# turn to df again
X_train_cat = pd.DataFrame(X_train_cat, columns=cat_cols, index=X_train.index)
X_test_cat = pd.DataFrame(X_test_cat, columns=cat_cols, index=X_test.index)

/Users/hedyeh/Ginger_Gradient/ds-ml-project/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [44]:
# scaling numerical features

scaler = MinMaxScaler()

X_train_num = scaler.fit_transform(X_train[numeric_features])
X_test_num = scaler.transform(X_test[numeric_features])

# save as df again
X_train_num = pd.DataFrame(X_train_num, columns=numeric_features, index=X_train.index)
X_test_num = pd.DataFrame(X_test_num, columns=numeric_features, index=X_test.index)

In [45]:
# create final train and validation sets 

X_train_final = pd.concat(
    [X_train_num, X_train_cat, X_train[binary_features]],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num, X_test_cat, X_test[binary_features]],
    axis=1
)

print(X_train_final.shape)
print(X_test_final.shape)

(630000, 35)
(270000, 35)


In [46]:
# check ordering
X_test_final = X_test_final[X_train_final.columns]

In [51]:
X_train_final.isna().sum()

Soil_pH                         0
Soil_Moisture                   0
Organic_Carbon                  0
Electrical_Conductivity         0
Temperature_C                   0
Humidity                        0
Rainfall_mm                     0
Sunlight_Hours                  0
Wind_Speed_kmh                  0
Field_Area_hectare              0
Previous_Irrigation_mm          0
Soil_Type_Loamy                 0
Soil_Type_Sandy                 0
Soil_Type_Silt                  0
Crop_Type_Maize                 0
Crop_Type_Potato                0
Crop_Type_Rice                  0
Crop_Type_Sugarcane             0
Crop_Type_Wheat                 0
Irrigation_Type_Drip            0
Irrigation_Type_Rainfed         0
Irrigation_Type_Sprinkler       0
Water_Source_Rainwater          0
Water_Source_Reservoir          0
Water_Source_River              0
Region_East                     0
Region_North                    0
Region_South                    0
Region_West                     0
Crop_Growth_St

### lets apply the best model we got from RandomizedSearchCV to the test set

### from notebook 02, we have:
Best GBC model: GradientBoostingClassifier(learning_rate=0.2, max_depth=5, max_features='sqrt',
                           min_samples_leaf=4, min_samples_split=10,
                           n_estimators=300, random_state=50)

In [52]:
best_gbc = GradientBoostingClassifier(learning_rate=0.2, max_depth=5, max_features='sqrt', min_samples_leaf=4, min_samples_split=10, n_estimators=300, random_state=RSEED)
best_gbc.fit(X_train_final, y_train)
y_pred_best_gbc = best_gbc.predict(X_test_final)

In [53]:
# inverse mapping (0,1,2 to strings)
inv_target_map = {0: "Low", 1: "Medium", 2: "High"}
y_pred_labels = pd.Series(y_pred_best_gbc).map(inv_target_map)

In [54]:
sample = pd.read_csv("data/sample_submission.csv")
print(sample.head())

       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low


In [55]:
# save the result
submission = sample.copy()
submission.iloc[:, 1] = y_pred_labels

submission.to_csv("submission.csv", index=False)

In [ ]:
# further final check
print(submission.shape)
print(submission.columns)
print(pd.Series(y_pred_best_gbc).value_counts())

(270000, 2)
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
Index(['id', 'Irrigation_Need'], dtype='object')
0    159886
1    101504
2      8610
Name: count, dtype: int64
